In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from argparse import Namespace
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# project imports
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from ml.utils.data_utils import prepare_dataset
from ml.models.lstm import LSTM
from ml.models.multi_step_lstm import MultiStepLSTM
from ml.models.seq2seq_lstm import Seq2SeqLSTM
from ml.models.transformer import TimeSeriesTransformer

In [2]:
# =============================
# 0) CONFIG
# =============================
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGETS = ['rnti_count', 'rb_down', 'rb_up', 'down', 'up']
H       = 6  # forecast horizon

# DATA PATHS
FULL_BASE_PATH  = '../dataset/full_dataset.csv'
CLU_BASE_PATH   = '../dataset/combined_with_cluster_feature.csv'
CLU_EXTRA_PATH  = '../dataset/combined_with_cluster_feature_with_extraData_26Aug.csv'

# CHECKPOINTS (trained on BASE data)
CKPT_BASE_T1   = "base_lstm_t1.pt"
CKPT_MULTI     = "multi_step_lstm.pt"
CKPT_S2S_MULTI = "seq2seq_lstm_multistep.pth"
CKPT_TRANS     = "transformer_multistep.pt"
CKPT_S2S_CLU   = "seq2seq_cluster_huber.pt"
CKPT_TRANS_CLU = "transformer_multistep_cluster.pt"

# CHECKPOINTS (trained on BASE + EXTRA data)
CKPT_S2S_CLU_EXTRA   = "seq2seq_cluster_with_extra_data_26Aug.pt"
CKPT_TRANS_CLU_EXTRA = "transformer_multistep_cluster_with_extra_data_26Aug.pt"

In [3]:
# =============================
# 1) HELPERS
# =============================
def mape(y_true, y_pred, eps=1e-8):
    denom = np.clip(np.abs(y_true), eps, None)
    return float(np.mean(np.abs((y_true - y_pred)/denom)) * 100.0)

def smape(y_true, y_pred, eps=1e-8):
    denom = np.clip((np.abs(y_true) + np.abs(y_pred)) / 2.0, eps, None)
    return float(np.mean(np.abs(y_true - y_pred) / denom) * 100.0)

def inverse_single_col(y_scaled, scaler, j):
    """
    Inverse-transform one target column j using fitted MinMax/Standard scaler.
    y_scaled can be 1D or 2D; returns numpy array of same shape in ORIGINAL units.
    """
    y_scaled = np.asarray(y_scaled)
    if hasattr(scaler, "min_") and hasattr(scaler, "scale_"):    # MinMax
        return (y_scaled - scaler.min_[j]) / scaler.scale_[j]
    if hasattr(scaler, "mean_") and hasattr(scaler, "scale_"):   # Standard
        return y_scaled * scaler.scale_[j] + scaler.mean_[j]
    raise ValueError("Unknown scaler type for inverse transform.")

def core_metrics_original_col(y_true_scaled, y_pred_scaled, scaler, j):
    """Compute MSE/RMSE/MAE/R2/NRMSE on ORIGINAL scale for a single target column j."""
    y_true = inverse_single_col(y_true_scaled, scaler, j)
    y_pred = inverse_single_col(y_pred_scaled, scaler, j)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    rng  = (np.max(y_true) - np.min(y_true)) + 1e-8
    nrmse = rmse / rng
    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2, "NRMSE": nrmse}, (y_true, y_pred)

def percent_metrics_from_pairs(y_true, y_pred):
    return {"MAPE%": mape(y_true, y_pred), "sMAPE%": smape(y_true, y_pred)}

def roll_base_lstm_to_horizon(base_model, X_init, steps, target_pos_in_x, device="cpu"):
    base_model.eval()
    x_win = torch.tensor(X_init, dtype=torch.float32, device=device)
    outs = []
    with torch.no_grad():
        for _ in range(steps):
            y_next = base_model(x_win, device=device)  # [N,T] (scaled)
            outs.append(y_next.unsqueeze(1))
            last_row = x_win[:, -1, :].clone()
            for k, pos in enumerate(target_pos_in_x):
                last_row[:, pos] = y_next[:, k]
            x_win = torch.cat([x_win[:, 1:, :], last_row.unsqueeze(1)], dim=1)
    return torch.cat(outs, dim=1).cpu().numpy()  # [N, steps, T] (scaled)

def infer_target_positions_from_data(X_test, y_t1_scaled):
    N, L, D = X_test.shape
    T = y_t1_scaled.shape[1]
    X_last = X_test[:, -1, :]
    pos, used = [], set()
    for i in range(T):
        yt = y_t1_scaled[:, i]
        yt = yt - yt.mean(); yt_std = yt.std() + 1e-12
        corrs = []
        for j in range(D):
            xj = X_last[:, j]
            xj = xj - xj.mean(); xj_std = xj.std() + 1e-12
            corr = float(np.mean((xj/xj_std) * (yt/yt_std)))
            corrs.append(abs(corr))
        for j in np.argsort(corrs)[::-1]:
            if j not in used:
                pos.append(int(j)); used.add(int(j)); break
    return pos if len(pos)==T else list(range(T))

def load_split(data_path, use_time_features=False):
    args = Namespace(
        data_path=data_path, targets=TARGETS, num_lags=10, forecast_steps=H,
        test_size=0.2, ignore_cols=None, identifier='District', nan_constant=0,
        x_scaler='minmax', y_scaler='minmax', outlier_detection=True,
        batch_size=128, cuda=torch.cuda.is_available(), seed=42,
        use_time_features=use_time_features
    )
    return prepare_dataset(args)

# ----- Eval blocks (UNSCALED) -----
def eval_strategy_A_block_UNSCALED(name, y_pred_scaled, y_true_scaled, scaler):
    """
    Build rows for Strategy A (t+1) on ORIGINAL scale.
    """
    rows = []
    for j, tgt in enumerate(TARGETS):
        core, (yt, yp) = core_metrics_original_col(y_true_scaled[:, j], y_pred_scaled[:, j], scaler, j)
        pct = percent_metrics_from_pairs(yt, yp)
        rows.append({"Strategy": "A_t+1", "Model": name, "Target": tgt, **core, **pct})
    return rows

def eval_strategy_B_block_UNSCALED(name, y_pred_scaled, y_true_scaled, scaler, tag="B"):
    """
    Build rows for Strategy B per-step + overall on ORIGINAL scale.
    """
    rows_steps, rows_over = [], []
    # per-step
    S = y_true_scaled.shape[1]
    for step in range(S):
        for j, tgt in enumerate(TARGETS):
            core, (yt, yp) = core_metrics_original_col(y_true_scaled[:, step, j], y_pred_scaled[:, step, j], scaler, j)
            pct = percent_metrics_from_pairs(yt, yp)
            rows_steps.append({
                "Strategy": f"{tag}_t+{step+1}", "Step": step+1, "Model": name, "Target": tgt, **core, **pct
            })
    # overall (flatten within each target)
    for j, tgt in enumerate(TARGETS):
        yt_all = y_true_scaled[:, :, j].reshape(-1)
        yp_all = y_pred_scaled[:, :, j].reshape(-1)
        core, (yt_o, yp_o) = core_metrics_original_col(yt_all, yp_all, scaler, j)
        pct = percent_metrics_from_pairs(yt_o, yp_o)
        rows_over.append({"Strategy": f"{tag}_overall", "Model": name, "Target": tgt, **core, **pct})
    return rows_steps, rows_over

In [4]:
# =============================
# 2) FAMILY 1: FULL (no cluster) – BASE DATA ONLY
# =============================
print("\n=== FAMILY 1: FULL (no cluster) – BASE DATA ONLY (UNSCALED METRICS) ===")
X_tr, y_tr, X_te, y_te, xsc_f, ysc_f, *_ = load_split(FULL_BASE_PATH)
N, L, D = X_te.shape; T = y_te.shape[2]
y_te_t1 = y_te[:, 0, :]  # scaled

# models
base_m = LSTM(input_dim=D, lstm_hidden_size=128, num_lstm_layers=2,
              lstm_dropout=0.0, layer_units=[128,64], num_outputs=T,
              matrix_rep=True, exogenous_dim=0).to(DEVICE)
base_m.load_state_dict(torch.load(CKPT_BASE_T1, map_location=DEVICE), strict=True)
base_m.eval()

basic_m = MultiStepLSTM(input_size=D, hidden_size=128, num_layers=1,
                        output_size=T, forecast_steps=H).to(DEVICE)
basic_m.load_state_dict(torch.load(CKPT_MULTI, map_location=DEVICE), strict=True)
basic_m.eval()

s2s_m = Seq2SeqLSTM(input_size=D, hidden_size=64, output_size=T,
                    forecast_steps=H, num_layers=1).to(DEVICE)
s2s_m.load_state_dict(torch.load(CKPT_S2S_MULTI, map_location=DEVICE), strict=True)
s2s_m.eval()

trans_m = TimeSeriesTransformer(input_size=D, output_size=T, forecast_steps=H,
                                d_model=128, nhead=4, num_encoder_layers=2,
                                num_decoder_layers=2, dim_feedforward=256,
                                dropout=0.1).to(DEVICE)
trans_m.load_state_dict(torch.load(CKPT_TRANS, map_location=DEVICE), strict=True)
trans_m.eval()

with torch.no_grad():
    xb = torch.tensor(X_te, dtype=torch.float32, device=DEVICE)
    base_t1_s = base_m(xb, device=DEVICE).cpu().numpy()          # [N,T] scaled
    basic_all = basic_m(xb).cpu().numpy()                         # [N,H,T] scaled
    s2s_all   = s2s_m(xb, teacher_forcing_ratio=0.0).cpu().numpy()
    trans_all = trans_m(xb).cpu().numpy()

# Strategy A – t+1 (UNSCALED)
rowsA = []
rowsA += eval_strategy_A_block_UNSCALED("Base LSTM (t+1)",            base_t1_s,        y_te_t1, ysc_f)
rowsA += eval_strategy_A_block_UNSCALED("Basic Multistep LSTM (t+1)", basic_all[:,0,:], y_te_t1, ysc_f)
rowsA += eval_strategy_A_block_UNSCALED("Seq2Seq LSTM (t+1)",         s2s_all[:,0,:],   y_te_t1, ysc_f)
rowsA += eval_strategy_A_block_UNSCALED("Transformer (t+1)",          trans_all[:,0,:], y_te_t1, ysc_f)
UNS_A_full_base = pd.DataFrame(rowsA)

# Strategy B – t+1..t+6 (UNSCALED)
TARGET_POS_IN_X = infer_target_positions_from_data(X_te, y_te_t1)
base_roll = roll_base_lstm_to_horizon(base_m, X_te, H, TARGET_POS_IN_X, device=DEVICE)  # scaled

rowsB_steps, rowsB_over = [], []
s, o = eval_strategy_B_block_UNSCALED("Base LSTM (rolled)", base_roll, y_te, ysc_f, tag="B(FULL-BASE)")
rowsB_steps += s; rowsB_over += o
s, o = eval_strategy_B_block_UNSCALED("Basic Multistep LSTM", basic_all, y_te, ysc_f, tag="B(FULL-BASE)")
rowsB_steps += s; rowsB_over += o
s, o = eval_strategy_B_block_UNSCALED("Seq2Seq LSTM", s2s_all, y_te, ysc_f, tag="B(FULL-BASE)")
rowsB_steps += s; rowsB_over += o
s, o = eval_strategy_B_block_UNSCALED("Transformer", trans_all, y_te, ysc_f, tag="B(FULL-BASE)")
rowsB_steps += s; rowsB_over += o
UNS_B_steps_full_base   = pd.DataFrame(rowsB_steps)
UNS_B_overall_full_base = pd.DataFrame(rowsB_over)



=== FAMILY 1: FULL (no cluster) – BASE DATA ONLY (UNSCALED METRICS) ===


In [5]:
# =============================
# 3) FAMILY 2: CLUSTERED – BASE DATA ONLY
# =============================
print("\n=== FAMILY 2: CLUSTERED – BASE DATA ONLY (UNSCALED METRICS) ===")
X_tr_c, y_tr_c, X_te_c, y_te_c, xsc_c, ysc_c, *_ = load_split(CLU_BASE_PATH)
Nc, Lc, Dc = X_te_c.shape; y_te_c_t1 = y_te_c[:, 0, :]

s2s_c = Seq2SeqLSTM(input_size=Dc, hidden_size=64, output_size=T, forecast_steps=H, num_layers=1).to(DEVICE)
s2s_c.load_state_dict(torch.load(CKPT_S2S_CLU, map_location=DEVICE), strict=True)
s2s_c.eval()

trans_c = TimeSeriesTransformer(input_size=Dc, output_size=T, forecast_steps=H,
                                d_model=128, nhead=4, num_encoder_layers=2,
                                num_decoder_layers=2, dim_feedforward=256,
                                dropout=0.1).to(DEVICE)
trans_c.load_state_dict(torch.load(CKPT_TRANS_CLU, map_location=DEVICE), strict=True)
trans_c.eval()

with torch.no_grad():
    xb = torch.tensor(X_te_c, dtype=torch.float32, device=DEVICE)
    s2s_c_all   = s2s_c(xb, teacher_forcing_ratio=0.0).cpu().numpy()
    trans_c_all = trans_c(xb).cpu().numpy()

rowsA = []
rowsA += eval_strategy_A_block_UNSCALED("Seq2Seq LSTM + Clusters (t+1)", s2s_c_all[:,0,:], y_te_c_t1, ysc_c)
rowsA += eval_strategy_A_block_UNSCALED("Transformer + Clusters (t+1)",  trans_c_all[:,0,:], y_te_c_t1, ysc_c)
UNS_A_cluster_base = pd.DataFrame(rowsA)

rowsB_steps, rowsB_over = [], []
s, o = eval_strategy_B_block_UNSCALED("Seq2Seq LSTM + Clusters", s2s_c_all, y_te_c, ysc_c, tag="B(CLU-BASE)")
rowsB_steps += s; rowsB_over += o
s, o = eval_strategy_B_block_UNSCALED("Transformer + Clusters", trans_c_all, y_te_c, ysc_c, tag="B(CLU-BASE)")
rowsB_steps += s; rowsB_over += o
UNS_B_steps_cluster_base   = pd.DataFrame(rowsB_steps)
UNS_B_overall_cluster_base = pd.DataFrame(rowsB_over)


=== FAMILY 2: CLUSTERED – BASE DATA ONLY (UNSCALED METRICS) ===


In [6]:
# =============================
# 4) FAMILY 3: CLUSTERED – BASE + EXTRA DATA
# =============================
print("\n=== FAMILY 3: CLUSTERED – BASE + EXTRA DATA (UNSCALED METRICS) ===")
X_tr_ce, y_tr_ce, X_te_ce, y_te_ce, xsc_ce, ysc_ce, *_ = load_split(CLU_EXTRA_PATH)
Ne, Le, De = X_te_ce.shape; y_te_ce_t1 = y_te_ce[:, 0, :]

s2s_ce = Seq2SeqLSTM(input_size=De, hidden_size=64, output_size=T, forecast_steps=H, num_layers=1).to(DEVICE)
s2s_ce.load_state_dict(torch.load(CKPT_S2S_CLU_EXTRA, map_location=DEVICE), strict=True)
s2s_ce.eval()

trans_ce = TimeSeriesTransformer(input_size=De, output_size=T, forecast_steps=H,
                                 d_model=128, nhead=4, num_encoder_layers=2,
                                 num_decoder_layers=2, dim_feedforward=256,
                                 dropout=0.1).to(DEVICE)
trans_ce.load_state_dict(torch.load(CKPT_TRANS_CLU_EXTRA, map_location=DEVICE), strict=True)
trans_ce.eval()

with torch.no_grad():
    xb = torch.tensor(X_te_ce, dtype=torch.float32, device=DEVICE)
    s2s_ce_all   = s2s_ce(xb, teacher_forcing_ratio=0.0).cpu().numpy()
    trans_ce_all = trans_ce(xb).cpu().numpy()

rowsA = []
rowsA += eval_strategy_A_block_UNSCALED("Seq2Seq LSTM + Clusters + Extra (t+1)", s2s_ce_all[:,0,:], y_te_ce_t1, ysc_ce)
rowsA += eval_strategy_A_block_UNSCALED("Transformer + Clusters + Extra (t+1)",  trans_ce_all[:,0,:], y_te_ce_t1, ysc_ce)
UNS_A_cluster_extra = pd.DataFrame(rowsA)

rowsB_steps, rowsB_over = [], []
s, o = eval_strategy_B_block_UNSCALED("Seq2Seq LSTM + Clusters + Extra", s2s_ce_all, y_te_ce, ysc_ce, tag="B(CLU-EXTRA)")
rowsB_steps += s; rowsB_over += o
s, o = eval_strategy_B_block_UNSCALED("Transformer + Clusters + Extra", trans_ce_all, y_te_ce, ysc_ce, tag="B(CLU-EXTRA)")
rowsB_steps += s; rowsB_over += o
UNS_B_steps_cluster_extra   = pd.DataFrame(rowsB_steps)
UNS_B_overall_cluster_extra = pd.DataFrame(rowsB_over)


=== FAMILY 3: CLUSTERED – BASE + EXTRA DATA (UNSCALED METRICS) ===


In [7]:
# =============================
# 5) SAVE (same names, prefixed "UNSCALED_")
# =============================
out_dir = '../dataset/capstone_results/'
os.makedirs(out_dir, exist_ok=True)

UNS_A_full_base.to_csv(os.path.join(out_dir, "UNSCALED_A_full_base.csv"), index=False)
UNS_B_steps_full_base.to_csv(os.path.join(out_dir, "UNSCALED_B_steps_full_base.csv"), index=False)
UNS_B_overall_full_base.to_csv(os.path.join(out_dir, "UNSCALED_B_overall_full_base.csv"), index=False)

UNS_A_cluster_base.to_csv(os.path.join(out_dir, "UNSCALED_A_cluster_base.csv"), index=False)
UNS_B_steps_cluster_base.to_csv(os.path.join(out_dir, "UNSCALED_B_steps_cluster_base.csv"), index=False)
UNS_B_overall_cluster_base.to_csv(os.path.join(out_dir, "UNSCALED_B_overall_cluster_base.csv"), index=False)

UNS_A_cluster_extra.to_csv(os.path.join(out_dir, "UNSCALED_A_cluster_extra.csv"), index=False)
UNS_B_steps_cluster_extra.to_csv(os.path.join(out_dir, "UNSCALED_B_steps_cluster_extra.csv"), index=False)
UNS_B_overall_cluster_extra.to_csv(os.path.join(out_dir, "UNSCALED_B_overall_cluster_extra.csv"), index=False)

print("\nSaved UNSCALED tables to:", out_dir)


Saved UNSCALED tables to: ../dataset/capstone_results/
